In [4]:
from datasets import load_dataset

dataset=load_dataset("SetFit/emotion")

Repo card metadata block was not found. Setting CardData to empty.


In [5]:
train_data=dataset["train"]
test_data=dataset["test"]

In [6]:
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained("bert-base-uncased")

In [7]:
# Tokenization function
def tokenize_function(batch):
    return tokenizer(batch["text"],padding="max_length",truncation=True)

# Apply tokenization
train_dataset = train_data.map(tokenize_function, batched=True)
test_dataset = test_data.map(tokenize_function, batched=True)

# Convert to PyTorch format
train_dataset.set_format(type="torch",columns=["input_ids", "attention_mask", "label"])

test_dataset.set_format(type="torch",columns=["input_ids", "attention_mask", "label"])

In [8]:
from transformers import AutoModelForSequenceClassification

#load bert model
base_model=AutoModelForSequenceClassification.from_pretrained("bert-base-uncased",num_labels=6)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
from peft import  get_peft_model, prepare_model_for_kbit_training, TaskType, LoraConfig

# Define LoRA Configuration
lora_config = LoraConfig(
    r=8,                 # Low-rank adaptation dimension , the more the value the better the performance but also more computation
    lora_alpha=32,       # Scaling factor
    lora_dropout=0.05,   # Dropout rate
    target_modules=["query", "value"]  # Apply LoRA to self-attention layers only , coz we are using bert which is transformer based model
)

# Prepare model for LoRA
base_model = prepare_model_for_kbit_training(base_model)

# Convert model into LoRA-enabled model
peft_model = get_peft_model(base_model, lora_config)

# Print trainable parameters
peft_model.print_trainable_parameters()


trainable params: 294,912 || all params: 109,781,766 || trainable%: 0.2686


In [10]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("GPU name:", torch.cuda.get_device_name(0))


Torch version: 2.7.1+cu118
CUDA available: True
GPU count: 1
GPU name: NVIDIA GeForce GTX 1650


In [13]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(  # configuration class that defines how training should happen
    output_dir="./model_checkpoints",   # Where to save model
    num_train_epochs=3,                 # Train for 3 epochs
    per_device_train_batch_size=16,     # 16 samples per GPU/CPU
    eval_strategy="epoch",              # Evaluate after every epoch
    save_strategy="epoch",              # Save model after each epoch
    logging_steps=10,                   # Log training metrics every 10 steps
    load_best_model_at_end=True,         # Automatically load best checkpoint
    fp16=False                            # Use mixed precision for faster training (if GPU supports it)
)

# A high-level class that automates training, evaluation, and saving models.
# It wraps around your model and dataset, handling:
# - Training loops
# - Evaluation during training
# - Model saving & checkpointing

trainer = Trainer(
    model=peft_model,          # LoRA fine-tuned model
    args=training_args,        # Training settings
    train_dataset=train_dataset,  # Training data
    eval_dataset=test_dataset,    # Test data
    tokenizer=tokenizer        # Tokenizer for processing text
)

trainer.train()


C:\Users\Abhishek Raj\AppData\Local\Temp\ipykernel_4780\1645111920.py:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,1.313000,1.165364
2,1.067100,1.015033
3,1.094000,0.963863


c:\Users\Abhishek Raj\Desktop\Machine_Learning\venv\lib\site-packages\transformers\utils\generic.py:255: FutureWarning: The input object of type 'Tensor' is an array-like implementing one of the corresponding protocols (`__array__`, `__array_interface__` or `__array_struct__`); but not a sequence (or 0-D). In the future, this object will be coerced as if it was first converted using `np.array(obj)`. To retain the old behaviour, you have to either modify the type 'Tensor', or assign to an empty array created with `np.empty(correct_shape, dtype=object)`.
  arr = np.array(obj)
c:\Users\Abhishek Raj\Desktop\Machine_Learning\venv\lib\site-packages\transformers\utils\generic.py:255: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  arr = np.array(obj)
'(MaxRetryError('HTTPSConn

TrainOutput(global_step=3000, training_loss=1.224702158610026, metrics={'train_runtime': 30726.4908, 'train_samples_per_second': 1.562, 'train_steps_per_second': 0.098, 'total_flos': 1.2673270775808e+16, 'train_loss': 1.224702158610026, 'epoch': 3.0})

In [35]:
model = trainer.model
model.eval()



PeftModel(
  (base_model): LoraModel(
    (model): BertForSequenceClassification(
      (bert): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSdpaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=76

In [38]:
device = next(model.parameters()).device

device


device(type='cuda', index=0)

In [41]:
new_inputs = {}

for k, v in inputs.items():
    new_inputs[k] = v.to(device)

inputs = new_inputs

# v.to(device) means moving each tensor to the device and k,v are key value pairs in the inputs dictionary
# k = 'input_ids', 'attention_mask', etc.
# v = the corresponding tensor

# 🌟🌟🌟🌟🌟
# here we are moving the inputs to the same device as model
# this is important for computation to happen on the same device
# training can happen on two devices cpu or gpu but inference should happen on the same device as model

In [ ]:
import torch

model.eval()

device = next(model.parameters()).device

tests = [
    "I absolutely love this, best thing ever",
    "This is horrible, I hate it so much",
    "Worst product in the world",
    "Fantastic, exceeded all expectations"
]

inputs = tokenizer(
    tests,
    return_tensors="pt",
    padding=True,
    truncation=True
)

# move inputs to same device as model
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

probs = torch.softmax(outputs.logits, dim=-1)
confidence, preds = torch.max(probs, dim=-1)

for text, pred, conf in zip(tests, preds, confidence):
    print(text)
    print(f" → class {pred.item()}, confidence {conf.item():.2f}")


I absolutely love this, best thing ever
 → class 1, confidence 0.64
This is horrible, I hate it so much
 → class 0, confidence 0.55
Worst product in the world
 → class 0, confidence 0.48
Fantastic, exceeded all expectations
 → class 1, confidence 0.64


I absolutely love this, best thing ever
 → class 1, confidence 0.57
This is horrible, I hate it so much
 → class 0, confidence 0.58
Worst product in the world
 → class 0, confidence 0.53
Fantastic, exceeded all expectations
 → class 1, confidence 0.58


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): lora.Linear(
                (base_layer): Linear(in_features=768, out_features=768, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_featu

ImportError: Using `bitsandbytes` 4-bit quantization requires the latest version of bitsandbytes: `pip install -U bitsandbytes`